# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


## Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

- we can  use multiple trigger in summarization :like token trigger messages trigger 

#### trigger based on Messages

In [2]:
from langchain_core.exceptions import LangChainException
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

### MESSAGES BASED SUMMARIZATION 

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
### run with thread id 
config = {"configurable":{"thread_id":"test-1"}}

In [4]:
### alternative data 
question = [
    "what is 2+2?",
    "what is 2*6?",
    "what is 7*7?",
    "what is 6/3?",
    "what is 3*4*3?",
    "what is 6/2?"
]
for q in question:
    responce = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages{responce}")
    print(f"MESSAGES:{len(responce['messages'])}")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Messages{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='234f67b0-f3a1-469f-a83e-c6b31ce30044'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05d2a-9a34-7742-bce7-97394421b5f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 28, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})]}
MESSAGES:2


GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 14.252911101s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

### trigger based on token size 

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str)->str:
    """search hotels - return long responce to use more tokens."""
    return f"hotel in{city}: 1. Grand Hotel - 5 star, $350/night, spa, pool, gym 2. City Inn - 4 star, $180/night, business center 3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("tokens",550),
            keep=("tokens",200)

        )
    ]
)
config = {"configurable":{"thread_id":"test-1"}}
def countTok(messages):
    total_chars =sum(len(str(m.content))for m in messages)
    return total_chars // 4 # 4 chars =1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = countTok(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~180 tokens, 8 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='730b21aa-2c95-495e-ae66-63bd02260882'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'e02eabe3-1626-46d0-b79c-dd4741c08e54': 'CpkCARFNMg8vBeszso8zvDajBqH27Xs0Ya1hfBRz4OO9j3il0wBMLDlKjRr74Tomv8DdD89jD8l2+UkR1dLegNoLvrMYAFq/7rVpQEbfw799SNp0LvDZVKgPK6D4cuP7+klZKvRAfvUXKXoiG1lsTZSBZwBBds42Dpl0DzH7RSBMgYLkvSo5Qmk8v691EBosPqNkwXDrCeZ3YkaFFPOqLpa2QbXIMjPBnhUATdo5S5te98SVCDXMv4ijcf64DVRG0+oM23hmNToCCV2xshsZVuO5NvnZeDNKaZ8fEhdbd7I3qyafgu7JNigLgNWszmEraOo1aaDflLWtWlWB09G2hVoHBzhswvOp20WKXUqA1uSLkw04f5H9U5MrR2k='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05c04-6624-7c31-8838-a82577432f46-0', tool_calls=[{'name': 'search_hotels', 'args': {'city':

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 21.960569758s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

 - above error error is nothing just notification of quota of gemini key is out 
### we also  do based on fraction 

In [5]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [8]:
import time

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


# Create agent
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",

            # Low fraction for testing
            trigger=("fraction", 0.005),

            # Keep a small portion after summarization
            keep=("fraction", 0.002),
        )
    ]
)


# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4


# Configuration for memory/thread
config = {
    "configurable": {
        "thread_id": "hotel-test-1"
    }
}


# Test cities
cities = [
    "Paris",
    "London",
    "Tokyo",
    "New York",
    "Dubai",
    "Singapore"
]


for city in cities:

    try:
        response = agent.invoke(
            {
                "messages": [
                    HumanMessage(content=f"Find hotels in {city}")
                ]
            },
            config=config
        )

        tokens = count_tokens(response["messages"])

        fraction = tokens / 128000

        print(
            f"{city}: ~{tokens} tokens "
            f"({fraction:.4%}), "
            f"{len(response['messages'])} messages"
        )

        print(response["messages"])
        print("-" * 80)

        # Prevent Groq rate limit
        time.sleep(4)

    except Exception as e:
        print(f"Error for {city}: {e}")

        # Wait longer if rate limit occurs
        print("Waiting 10 seconds before continuing...")
        time.sleep(10)

Paris: ~96 tokens (0.0750%), 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='356f9e47-0fca-463b-bf2a-1b7f34826122'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_64bdd765-ec07-4b76-af79-fa06486e8c64', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 121, 'total_tokens': 160, 'completion_time': 0.04238589, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.009287726, 'prompt_tokens_details': None, 'queue_time': 0.330191886, 'total_time': 0.051673616}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05d2f-d38b-7fe3-a551-e3b84a2fa511-0', too

# Human In the Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email(email_id:str)->str:
    """MOCK FUNCTION TO READ EMAIL AND ITS ID"""
    return f"Email content for ID  {email_id}"
def send_email_tool(reciptent:str,subject:str,body:str):
    """mock function to send email"""
    return f"emial sent to {reciptent},with subject {subject}"

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[read_email, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": [
                        "approve",
                        "edit",
                        "reject"
                    ]
                },
                "read_email": False
            }
        )
    ]
)

In [15]:
from langchain_core.messages import content
from langgraph.store.base import Result
config = {"configurable":{"thread_id":"test-approve"}}

#step1 
result = agent.invoke(
    {"messages":[HumanMessage(content="send message to vishal@gmail.com with subject 'hello' and body 'how are you' ")]},
    config=config
)

In [16]:
result

{'messages': [HumanMessage(content="send message to vishal@gmail.com with subject 'hello' and body 'how are you' ", additional_kwargs={}, response_metadata={}, id='0bed5411-0601-4c45-a0ee-ffe0a8f2c97e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_302e8b17-8312-49b7-9fae-0d8eb6e6ad6e', 'function': {'arguments': '{"body":"how are you","reciptent":"vishal@gmail.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 176, 'total_tokens': 225, 'completion_time': 0.054918915, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008541967, 'prompt_tokens_details': None, 'queue_time': 0.333549069, 'total_time': 0.063460882}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq

In [18]:
# Step 2: Approve
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

print(f" Result: {result['messages'][-1].content}")

 Result: ✅ Email sent to vishal@gmail.com with subject “hello” and body “how are you”.


 - simlarly we can do for reject and edit 